# 01 — Prepare data (CPU notebook)

**Settings (right sidebar):** Accelerator = **None**, Internet = **On**, Persistence = Files only.

Streams ~800M tokens of Python from `codeparrot/codeparrot-clean-train`, tokenizes with the
codeparrot 32k BPE tokenizer and writes `train.bin` / `val.bin` (uint16) to `/kaggle/working`.
Takes roughly 30–60 min. **Uses 0 GPU hours.**

When it finishes: **Save & Run All**, then on the finished version open the *Output* tab →
**New Dataset** → name it `pygpt-python-tokens`. Notebook 02 reads from that dataset.

In [ ]:
# --- Get the code into /kaggle/working/pygpt ---------------------------------
# Option A: public GitHub repo (set REPO_URL). Option B: a Kaggle Dataset named
# "pygpt-code" containing the repo folder (upload the zip -> Datasets -> New Dataset).
REPO_URL = ""            # e.g. "https://github.com/<you>/pygpt.git"
import os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.exists("pygpt"):
    shutil.rmtree("pygpt")
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pygpt"], check=True)
else:
    src = "/kaggle/input/pygpt-code"
    cand = [os.path.join(src, "pygpt"), src]
    for c in cand:
        if os.path.exists(os.path.join(c, "model.py")):
            shutil.copytree(c, "pygpt"); break
    else:
        raise SystemExit("set REPO_URL or add the 'pygpt-code' dataset as input")
print(sorted(os.listdir("pygpt")))

In [ ]:
%pip install -q datasets
import datasets, transformers; print("datasets", datasets.__version__, "transformers", transformers.__version__)

In [ ]:
# Quick test first (2M tokens, ~1 min). Set TRAIN_TOKENS to 800_000_000 for the real run.
TRAIN_TOKENS = 800_000_000
VAL_TOKENS   = 5_000_000
!cd /kaggle/working/pygpt && python data/prepare_data.py --out_dir /kaggle/working \
    --train_tokens {TRAIN_TOKENS} --val_tokens {VAL_TOKENS}

In [ ]:
import json, os
print(json.dumps(json.load(open("/kaggle/working/meta.json")), indent=2))
for f in ("train.bin", "val.bin"):
    print(f, f"{os.path.getsize('/kaggle/working/'+f)/2**20:,.0f} MB")
print(open("/kaggle/working/sample.txt").read()[:1500])

### Next
1. Confirm the sample above looks like real Python.
2. Output tab → **New Dataset** → `pygpt-python-tokens` (contains `train.bin`, `val.bin`, `meta.json`, `tokenizer/`).
3. Open Notebook 02.